In [1]:
# Scenarios Setup
# -------------------------------

import pickle
from pathlib import Path

import numpy as np

from app.analysis.distributions import Constant
from app.analysis.psa.parameter_resolver import ParameterResolver
from app.analysis.psa.parameters import Parameter
from app.analysis.psa.sampler import PSASampler
from app.domain.enums import HealthStates, Regime
from app.domain.inputs import ModelInput
from app.domain.scenario import Scenario, ScenarioBundle
from app.domain.worker import worker_function
from app.notebook_tools.parameter_sets import HemophiliaParamRepo
from app.notebook_tools.scenario_helpers import (
    define_scenario_extension,
    get_tornado_ranges,
    insert_scenario,
)
from app.persistence.context import ModelContext
from engine.chains import Chain
from utils.logging import setup_root_logger
from utils.path_utils import get_project_root

logger = setup_root_logger()
context = ModelContext.load()
seed = context.simulation.environment.seed
sample_size = context.simulation.psa.development

root = get_project_root()

cost_unit = context.costs.currencies[0].code
per_unit_cost = context.costs.costs[0].pricing.per_unit[cost_unit]


# Helper functions
def mean(value):
    return np.mean(value)


def low(value):
    return np.percentile(value, 2.5)


def high(value):
    return np.percentile(value, 97.5)


param_repo = HemophiliaParamRepo(root=root, cache_path=Path("app/cache/samples.pkl"), context=context)
with open(param_repo.root / param_repo.cache_path, "rb") as f:
    samples = pickle.load(f)

# on_demand base scenario
owsa_parameters = param_repo.load_owsa_parameters()
psa_parameters = param_repo.load_psa_parameters()
owsa_keys = param_repo.ows_params_keys

tornado_ranges = get_tornado_ranges(psa_parameters, owsa_keys, sample_size, seed)


scenarios = []


bayesian_scenario = [
    # OWSA _ EARLY
    Scenario(name="early on_demand bayesian", regime=Regime.ON_DEMAND, overrides={}),
    Scenario(
        name="early prophylaxis bayesian",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(value=10 * 52)),
            "bleeding_rate": Parameter(
                Constant(mean(samples["prophylaxis"]["bayesian"]))
            ),
        },
    ),
    # OWSA _ PROPHYLAXIS
    Scenario(
        name="lifetime on_demand bayesian",
        regime=Regime.ON_DEMAND,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
        },
    ),
    Scenario(
        name="lifetime prophylaxis bayesian",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "bleeding_rate": Parameter(
                Constant(mean(samples["prophylaxis"]["bayesian"]))
            ),
        },
    ),
]

# ---------------------------
#           OWSA
# ---------------------------
# Define low _ high scenarios
annual_bleeding_rate_scenarios = [
    Scenario(
        name="early on_demand bayesian abr_low",
        regime=Regime.ON_DEMAND,
        overrides={
            "bleeding_rate": Parameter(Constant(low(samples["on_demand"]["bayesian"]))),
        },
    ),
    Scenario(
        name="early prophylaxis bayesian abr_low",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(value=10 * 52)),
            "bleeding_rate": Parameter(
                Constant(low(samples["prophylaxis"]["bayesian"]))
            ),
        },
    ),
    # OWSA _ PROPHYLAXIS
    Scenario(
        name="lifetime on_demand bayesian abr_low",
        regime=Regime.ON_DEMAND,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "bleeding_rate": Parameter(Constant(low(samples["on_demand"]["bayesian"]))),
        },
    ),
    Scenario(
        name="lifetime prophylaxis bayesian abr_low",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "bleeding_rate": Parameter(
                Constant(low(samples["prophylaxis"]["bayesian"]))
            ),
        },
    ),
    Scenario(
        name="early on_demand bayesian abr_high",
        regime=Regime.ON_DEMAND,
        overrides={
            "bleeding_rate": Parameter(
                Constant(high(samples["on_demand"]["bayesian"]))
            ),
        },
    ),
    Scenario(
        name="early prophylaxis bayesian abr_high",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(value=10 * 52)),
            "bleeding_rate": Parameter(
                Constant(high(samples["prophylaxis"]["bayesian"]))
            ),
        },
    ),
    # OWSA _ PROPHYLAXIS
    Scenario(
        name="lifetime on_demand bayesian abr_high",
        regime=Regime.ON_DEMAND,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "bleeding_rate": Parameter(
                Constant(high(samples["on_demand"]["bayesian"]))
            ),
        },
    ),
    Scenario(
        name="lifetime prophylaxis bayesian abr_high",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "bleeding_rate": Parameter(
                Constant(high(samples["prophylaxis"]["bayesian"]))
            ),
        },
    ),
]

weight_scenarios = [
    Scenario(
        name="early on_demand bayesian weight_low",
        regime=Regime.ON_DEMAND,
        overrides={
            "weight_factor": Parameter(distribution=Constant(value=0.9)),
        },
    ),
    Scenario(
        name="early prophylaxis bayesian weight_low",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(value=10 * 52)),
            "weight_factor": Parameter(distribution=Constant(value=0.9)),
        },
    ),
    # OWSA _ PROPHYLAXIS
    Scenario(
        name="lifetime on_demand bayesian weight_low",
        regime=Regime.ON_DEMAND,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "weight_factor": Parameter(distribution=Constant(value=0.9)),
        },
    ),
    Scenario(
        name="lifetime prophylaxis bayesian weight_low",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "weight_factor": Parameter(distribution=Constant(value=0.9)),
        },
    ),
    Scenario(
        name="early on_demand bayesian weight_high",
        regime=Regime.ON_DEMAND,
        overrides={
            "weight_factor": Parameter(distribution=Constant(value=1.1)),
        },
    ),
    Scenario(
        name="early prophylaxis bayesian weight_high",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(value=10 * 52)),
            "weight_factor": Parameter(distribution=Constant(value=1.1)),
        },
    ),
    # OWSA _ PROPHYLAXIS
    Scenario(
        name="lifetime on_demand bayesian weight_high",
        regime=Regime.ON_DEMAND,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "weight_factor": Parameter(distribution=Constant(value=1.1)),
        },
    ),
    Scenario(
        name="lifetime prophylaxis bayesian weight_high",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "weight_factor": Parameter(distribution=Constant(value=1.1)),
        },
    ),
]


scenarios.extend(bayesian_scenario)
scenarios.extend(weight_scenarios)
scenarios.extend(annual_bleeding_rate_scenarios)

for key in owsa_keys:
    for n in ["low", "high"]:
        insert_scenario(
            scenarios=scenarios,
            pair=define_scenario_extension(
                scenarios=bayesian_scenario,
                extensions={
                    f"{key}_{n}": {
                        key: Constant(value=tornado_ranges[key][n]),
                    }
                },
            ),
        )

# Discounting scenario
insert_scenario(
    scenarios=scenarios,
    pair=define_scenario_extension(
        scenarios=bayesian_scenario,
        extensions={
            "is_discounting": {
                "benefits_discount_rate": Constant(value=context.simulation.discounting.utility_rate_annual),
                "costs_discount_rate": Constant(value=context.simulation.discounting.cost_rate_annual),
            },
        },
    ),
)

g++ not available, if using conda: `conda install gxx`


In [2]:
chains = []
states = [state.value for state in HealthStates]

# NOTE:
# Identity chain uses to introduce the states and matrix shape for further runtime calculation of transition matrix per individual patient.
# This matrix will be updated on patients growth adjusting with new transitions probabilities as the patients natural mortality rate changes over time.

identity_chain = Chain(
    name="main",
    states=states,
    matrix=np.eye(
        N=len(states), M=len(states), dtype=np.float64
    ),  # Identity matrix (Cubic)
)
print(f"\n{identity_chain.matrix} \n Identity matrix")
chains.append(identity_chain)


[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]] 
 Identity matrix


In [3]:
from utils import stable_hash

bundles: list[ScenarioBundle[ModelInput]] = []

for scenario in scenarios:
    scenario_seed = stable_hash(seed, scenario.name)
    scenario_params = scenario.apply_overrides(owsa_parameters)

    sampler = PSASampler(scenario_params, seed=scenario_seed)
    raw_samples = sampler.sample(sample_size)
    resolved_samples = ParameterResolver.resolve_samples(raw_samples)

    inputs = [
        ParameterResolver.build_single(resolved_samples, i) for i in range(sample_size)
    ]

    bundles.append(
        ScenarioBundle(
            scenario=scenario,
            inputs=inputs,
        )
    )

    logger.info(
        "Generated %d model inputs, bundled",
        len(inputs),
        extra={"scenario": scenario.name},
    )

[19:11:06] INFO     Generated 1000 model inputs, bundled | scenario=early on_demand bayesian
[19:11:06] INFO     Generated 1000 model inputs, bundled | scenario=early prophylaxis bayesian
[19:11:06] INFO     Generated 1000 model inputs, bundled | scenario=lifetime on_demand bayesian
[19:11:06] INFO     Generated 1000 model inputs, bundled | scenario=lifetime prophylaxis bayesian
[19:11:06] INFO     Generated 1000 model inputs, bundled | scenario=early on_demand bayesian weight_low
[19:11:06] INFO     Generated 1000 model inputs, bundled | scenario=early prophylaxis bayesian weight_low
[19:11:06] INFO     Generated 1000 model inputs, bundled | scenario=lifetime on_demand bayesian weight_low
[19:11:06] INFO     Generated 1000 model inputs, bundled | scenario=lifetime prophylaxis bayesian weight_low
[19:11:06] INFO     Generated 1000 model inputs, bundled | scenario=early on_demand bayesian weight_high
[19:11:06] INFO     Generated 1000 model inputs, bundled | scenario=early prophylaxis b

In [4]:
from app.notebook_tools.scenario_runner import run_scenarios_in_batches

# run and write parquet files per pair
run_scenarios_in_batches(
    bundles=bundles,
    context=context,
    identity_chain=identity_chain,
    worker_function=worker_function,
    batch_size=4,
    output_dir=root / "app/cache" / "owsa" / "parquet",
    temp_dir=root / "app/cache" / "owsa" / "parquet_temp",
    options={
        "use_cached_temp": True,
    },
    engine="pathos",
)

[19:11:08] INFO     Starting batch runner with 124 scenario, batch size: 4, engine: pathos
[19:11:08] INFO     No cached temp parquet found in C:\Users\Jino\Desktop\University\NOKI\markov-chain-3-14\cache\owsa\parquet_temp, regenerating batches.


[19:13:38] INFO     Saved temp batch 0 to C:\Users\Jino\Desktop\University\NOKI\markov-chain-3-14\cache\owsa\parquet_temp\batch_0.parquet
[19:13:38] INFO     Batch 1/31 complete (4 scenarios processed, 120 remaining). Elapsed=150.1s, avg_batch=150.1s, est_remaining=4502.6s


[19:15:52] INFO     Saved temp batch 1 to C:\Users\Jino\Desktop\University\NOKI\markov-chain-3-14\cache\owsa\parquet_temp\batch_1.parquet
[19:15:52] INFO     Batch 2/31 complete (8 scenarios processed, 116 remaining). Elapsed=284.7s, avg_batch=142.3s, est_remaining=4127.6s


[19:18:04] INFO     Saved temp batch 2 to C:\Users\Jino\Desktop\University\NOKI\markov-chain-3-14\cache\owsa\parquet_temp\batch_2.parquet
[19:18:04] INFO     Batch 3/31 complete (12 scenarios processed, 112 remaining). Elapsed=416.8s, avg_batch=138.9s, est_remaining=3889.7s


[19:20:20] INFO     Saved temp batch 3 to C:\Users\Jino\Desktop\University\NOKI\markov-chain-3-14\cache\owsa\parquet_temp\batch_3.parquet
[19:20:20] INFO     Batch 4/31 complete (16 scenarios processed, 108 remaining). Elapsed=552.5s, avg_batch=138.1s, est_remaining=3729.5s


...

[20:13:27] INFO     Saved temp batch 27 to C:\Users\Jino\Desktop\University\NOKI\markov-chain-3-14\cache\owsa\parquet_temp\batch_27.parquet
[20:13:27] INFO     Batch 28/31 complete (112 scenarios processed, 12 remaining). Elapsed=3739.6s, avg_batch=133.6s, est_remaining=400.7s


[20:15:23] INFO     Saved temp batch 28 to C:\Users\Jino\Desktop\University\NOKI\markov-chain-3-14\cache\owsa\parquet_temp\batch_28.parquet
[20:15:23] INFO     Batch 29/31 complete (116 scenarios processed, 8 remaining). Elapsed=3855.0s, avg_batch=132.9s, est_remaining=265.9s


[20:17:29] INFO     Saved temp batch 29 to C:\Users\Jino\Desktop\University\NOKI\markov-chain-3-14\cache\owsa\parquet_temp\batch_29.parquet
[20:17:29] INFO     Batch 30/31 complete (120 scenarios processed, 4 remaining). Elapsed=3981.3s, avg_batch=132.7s, est_remaining=132.7s


[20:19:48] INFO     Saved temp batch 30 to C:\Users\Jino\Desktop\University\NOKI\markov-chain-3-14\cache\owsa\parquet_temp\batch_30.parquet
[20:19:48] INFO     Batch 31/31 complete (124 scenarios processed, 0 remaining). Elapsed=4120.3s, avg_batch=132.9s, est_remaining=0.0s
[20:19:48] INFO     Combining 31 temp batch files into a DF for pairing
[20:19:49] INFO     Saved combined results checkpoint to C:\Users\Jino\Desktop\University\NOKI\markov-chain-3-14\cache\owsa\parquet\all_results_combined.parquet
[20:19:49] INFO     Saved Parquet for pair early on_demand bayesian vs early prophylaxis bayesian -> C:\Users\Jino\Desktop\University\NOKI\markov-chain-3-14\cache\owsa\parquet\early_on_demand_bayesian_vs_early_prophylaxis_bayesian.parquet
[20:19:49] INFO     Saved Parquet for pair lifetime on_demand bayesian vs lifetime prophylaxis bayesian -> C:\Users\Jino\Desktop\University\NOKI\markov-chain-3-14\cache\owsa\parquet\lifetime_on_demand_bayesian_vs_lifetime_prophylaxis_bayesian.parquet
[2

[WindowsPath('C:/Users/Jino/Desktop/University/NOKI/markov-chain-3-14/cache/owsa/parquet/early_on_demand_bayesian_vs_early_prophylaxis_bayesian.parquet'),
 WindowsPath('C:/Users/Jino/Desktop/University/NOKI/markov-chain-3-14/cache/owsa/parquet/lifetime_on_demand_bayesian_vs_lifetime_prophylaxis_bayesian.parquet'),
 WindowsPath('C:/Users/Jino/Desktop/University/NOKI/markov-chain-3-14/cache/owsa/parquet/early_on_demand_bayesian_weight_low_vs_early_prophylaxis_bayesian_weight_low.parquet'),
 WindowsPath('C:/Users/Jino/Desktop/University/NOKI/markov-chain-3-14/cache/owsa/parquet/lifetime_on_demand_bayesian_weight_low_vs_lifetime_prophylaxis_bayesian_weight_low.parquet'),
 WindowsPath('C:/Users/Jino/Desktop/University/NOKI/markov-chain-3-14/cache/owsa/parquet/early_on_demand_bayesian_weight_high_vs_early_prophylaxis_bayesian_weight_high.parquet'),
 WindowsPath('C:/Users/Jino/Desktop/University/NOKI/markov-chain-3-14/cache/owsa/parquet/lifetime_on_demand_bayesian_weight_high_vs_lifetime_prop

In [5]:
import pandas as pd

df = pd.read_parquet(
    root / "app/cache" / "owsa" / "parquet" / "all_results_combined.parquet"
)
# Each scenario has `sample_size` rows, so divide total rows by sample_size to get scenario count
scenario_count = int(df.shape[0] / sample_size)
# Each pair has 2 scenarios (on-demand and prophylaxis), so divide by 2 to get pair count
scenario_pair_count = scenario_count // 2
# Each parameter scenario has 4 variations (base, low, high for both on-demand and prophylaxis), so divide by 4 to get unique parameter scenario count
scenario_param_count = scenario_count // 4
logger.info(
    "Total scenarios: %d, Total pairs: %d, Unique parameter scenarios: %d",
    scenario_count,
    scenario_pair_count,
    scenario_param_count,
)

[20:19:51] INFO     Total scenarios: 124, Total pairs: 62, Unique parameter scenarios: 31
